In [ ]:
import marimo as mo

# Session 5: Retrieval Variants

*One worker, one decision, one person who has to sign it.*

Nadia is a second-line support engineer on the service desk of a regional
broadband and mobile operator. A ticket comes in, and the answer is never
in one place: it is in **two thousand resolved tickets, eighty runbooks,
the technicians' field notes**, and two live systems — the CRM and the
network monitor. She wants an assistant that reads a new ticket and posts
a diagnosis on it, with the evidence. Her manager runs the desk, owns its
budget and its four-hour SLA, and has told her what he needs before he
signs off on any of it: **one page.**

| What the page has to show | Where this notebook produces it |
| --- | --- |
| **How often is it right?** Complete diagnoses, not "something relevant came back" | Tasks 1, 5, 7 — `complete_at_k` |
| **What does it cost?** Per ticket at query time, once for the corpus — and what needs approving, because a vendor review is a cost with a calendar | Task 3's embedding call, Task 7's per-query reranker, Task 9's *setup* column |
| **How long does a ticket wait?** The SLA is four hours; the assistant gets seconds of it | Task 7's latency budget, Task 8's cost table |
| **What would change your mind?** | Task 9's six questions |

This notebook is Nadia building that page. Dense retrieval and BM25 from
scratch in numpy, hybrid fusion, a cross-encoder reranker, and the two
tools an agent needs to search — every one scored against labelled answers
on one real corpus. The output is not a technique. It is a page with four
numbers on it and the reasons behind them, which is the only thing her
manager can sign.

---

## Table of Contents

- Task 1: The corpus, the labels, and a ceiling you have already hit
- Task 2: Chunking, and the unit you are really retrieving
- ❓ Question #1
- Task 3: Dense retrieval, from scratch
- Task 4: BM25, from scratch
- Task 5: Hybrid
- ❓ Question #2
- Task 6: The variable that matters more than the technique
- Task 7: Reranking, which is the fix
- ❓ Question #3
- Task 8: Agentic retrieval, when one shot is not enough
- Task 9: Choose, and say why
- ❓ Question #4
- Conclusion
- Additional Context
- Bonus Tasks

> 📓 **Not covered here:** generating an answer from what was retrieved,
> and handing the two search tools to a model so it drives the loop itself.
> Both are the challenge, and the week's reading covers what they need.
> Nadia's corpus is described, not shipped — the numbers below come from a
> corpus with labels, which hers does not have yet.

## 📖 Official documentation

Everything this notebook uses, from the people who maintain it.

| Tool | What you use it for here | Docs |
| --- | --- | --- |
| numpy | cosine, BM25, RRF and every ranking in this file | [numpy.org/doc](https://numpy.org/doc/stable/) |
| LiteLLM | `embedding()` in the Bonus Task, against your own endpoint | [docs.litellm.ai — embeddings](https://docs.litellm.ai/docs/embedding/supported_embedding) |
| HotpotQA | the corpus and its labels | [hotpotqa.github.io](https://hotpotqa.github.io/) |
| BAAI/bge-reranker-v2-m3 | the cross-encoder whose scores ship with the corpus | [huggingface.co/BAAI/bge-reranker-v2-m3](https://huggingface.co/BAAI/bge-reranker-v2-m3) |

---

Almost everyone learns one retrieval technique, uses it everywhere, and
concludes it works — because on any corpus, something always comes back and it
usually looks plausible. Nadia's first assistant did exactly that: embed
every ticket, take the top three, and the diagnoses read well. Then a
senior engineer noticed one recommending a modem swap for a fault that was
in the street cabinet.

This session gives you a corpus with **labelled answers**, so "looks
plausible" stops being the standard. You will measure several techniques
against the same questions and work out which one you would actually ship.

**The corpus is Wikipedia, not support tickets, and that is deliberate.**
Nadia's tickets have no labels yet — nobody has written down which record
answers which question — and without labels there is no first column on
the page. This corpus has labels, and nobody wrote it to make a technique
look good.
Each task ends with what the measurement means on her corpus — and, in the
challenge, on yours.

Nobody is going to tell you the ranking in advance. The corpus was not written
for this lesson, and several of the results contradict advice you can find in
a hundred blog posts.

**Estimated time:** 45–60 minutes

---
## Setup

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(mo.notebook_dir()).parent.parent))
from helpers import nb, ui

CFG = nb.bootstrap()
DATA = Path(mo.notebook_dir()) / "data"

def read_jsonl(path):
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

DOCS = read_jsonl(DATA / "corpus.jsonl")
QUESTIONS = read_jsonl(DATA / "questions.jsonl")
TITLES = [doc["title"] for doc in DOCS]
TEXTS = [doc["title"] + "\n" + doc["text"] for doc in DOCS]

print(
    f"✅ {len(DOCS)} paragraphs, {sum(len(text.split()) for text in TEXTS):,} words"
)
print(f"✅ {len(QUESTIONS)} questions, with labelled answers")

---
## Task 1 — The corpus, the labels, and a ceiling you have already hit

Real Wikipedia paragraphs from [HotpotQA](https://hotpotqa.github.io/), a
multi-hop question-answering dataset. Nobody wrote them for this lesson, which
is the point — a corpus authored to demonstrate a retrieval failure will
always demonstrate it.

Each question ships with ten candidate paragraphs: **two that contain the
answer and eight plausible distractors**, chosen by a retrieval system to be
hard rather than by us to be easy. Pool them and you have the corpus you are
about to search.

What makes this different from the usual retrieval demo is the labels.
`gold_titles` records which paragraphs actually answer each question, so
*"did retrieval work?"* stops being a judgement call.

In [ ]:
ui.table(
    [
        {
            "question": question["question"][:62]
            + ("…" if len(question["question"]) > 62 else ""),
            "answer": question["answer"][:26],
            "kind": question["kind"],
            "paragraphs needed": len(question["gold_titles"]),
        }
        for question in QUESTIONS
    ],
    title="The questions, and how many paragraphs each one needs",
)

Read the last column before writing any code.

**Every question needs two paragraphs.** One names an entity, the other
describes it. *"The director of Big Stone Gap is based in what New York
city?"* needs the film's page to learn who directed it and the director's page
to learn where she lives. Neither alone contains the answer.

So you already know something, without having retrieved anything:

> **A top-1 retriever scores zero here.** Not "poorly" — zero. It returns one
> paragraph, every question needs two, and no ranking function changes that.

That is not the dataset being unfair. It is a property of questions that
combine information, which is most of the questions people actually ask. The
standard pipeline — embed, take the top *k*, stuff the prompt — has no way to
express *"I found half the answer"*. It hands the model one paragraph and lets
it guess the rest, confidently.

**Nadia's questions have exactly this shape.** *"Why does this customer's
fibre drop every evening?"* needs the new ticket, which names the cabinet
port `OLT-17`, and the resolved ticket from last month on the same port,
which names the fix — a re-splice at cabinet `C-204`. Neither alone is the
answer. Her first assistant took the top three by similarity, got three
tickets about *"drops in the evening"* from three different cabinets, and
recommended a modem swap — fluently. That is the diagnosis the senior
engineer caught, and it is why the measure for the rest of this session is
not "did something relevant come back" but **how many complete answers you
can assemble, and at what cost.**

In [ ]:
ui.table(
    [
        {
            "title": doc["title"],
            "sentences": len(doc["sentences"]),
            "opening": doc["text"][:86] + "…",
        }
        for doc in DOCS[:6]
    ],
    title="Six of the hundred paragraphs",
)

---
## Task 2 — Chunking, and the unit you are really retrieving

Retrieval does not search documents. It searches whatever unit you decided to
embed — a decision usually made early, quietly, and never revisited, which is
why it causes most "retrieval returns nothing useful" reports.

This corpus arrives pre-chunked, one paragraph per record. That is a
reasonable default and worth interrogating rather than accepting, and the
labels let us interrogate it: each question names the **individual sentences**
that carry the answer, not just the paragraph.

Nadia's records arrive the same way — one ticket, one runbook, one field
note per row — and the boundary was chosen by whoever designed the
ticketing system, not by her. A resolved ticket is six hundred words of
customer back-and-forth, of which the resolution is one line at the end.

In [ ]:
BY_TITLE = {doc["title"]: doc for doc in DOCS}

def supporting_sentences(question: dict) -> list[str]:
    """The exact sentences the dataset says carry the answer."""
    out = []
    for ref in question["supporting"]:
        doc = BY_TITLE.get(ref["title"])
        if doc and ref["sentence"] < len(doc["sentences"]):
            out.append(doc["sentences"][ref["sentence"]].strip())
    return out

def answer_signal(question) -> float:
    gold_words = sum(
        len(BY_TITLE[title]["text"].split())
        for title in question["gold_titles"]
        if title in BY_TITLE
    )
    evidence_words = sum(
        len(sentence.split()) for sentence in supporting_sentences(question)
    )
    return evidence_words / gold_words if gold_words else 0.0

ui.table(
    [
        {
            "question": question["question"][:42] + "…",
            "gold paragraphs, words": sum(
                len(BY_TITLE[title]["text"].split())
                for title in question["gold_titles"]
                if title in BY_TITLE
            ),
            "sentences that matter, words": sum(
                len(sentence.split()) for sentence in supporting_sentences(question)
            ),
            "signal": f"{answer_signal(question):.0%}",
        }
        for question in QUESTIONS[:5]
    ],
    title="How much of a retrieved paragraph is actually the answer",
)
print(
    f"median across all {len(QUESTIONS)} questions: "
    f"{np.median([answer_signal(question) for question in QUESTIONS]):.0%}"
)

Most of what you retrieve is not the answer. That is the chunking tradeoff as a
number, and it cuts both ways:

| Smaller chunks | Larger chunks |
| --- | --- |
| Higher signal — less irrelevant text in the prompt | The sentence keeps the context that makes it interpretable |
| Cheaper per retrieved item | Fewer boundaries to get wrong |
| A pronoun with no antecedent: *"She was appointed in 1969"* | You pay for words that answer nothing |

Cut this corpus to single sentences and *"She was appointed Chief of
Protocol"* retrieves beautifully and tells the model nothing about who "she"
is. That is the failure worth remembering, because **the retrieval metric goes
up while the answers get worse** — and you will be looking at the metric.

> **The practical rule:** chunk at a boundary the author already chose —
> paragraph, section, list item — before inventing your own. Then measure.
> There is no default chunk size that survives contact with a real corpus.

For Nadia that boundary is the ticket's own fields — *Symptom*,
*Diagnosis*, *Resolution* — which the ticketing system already gives her.
Cutting a *Resolution* away from its *Symptom* is how *"re-spliced at the
cabinet"* retrieves beautifully and no longer says for which fault, or on
which port.

---
### ❓ Question #1

Your own corpus arrives as rows from a generator, one record each. In one
to three sentences: what boundary did the generator already choose for
you, and what would you have to *measure* — not assume — before deciding
to cut inside it? The signal table above is the shape of that
measurement.

**Answer:**

---
## Task 3 — Dense retrieval, from scratch

An embedding model maps text to a vector so that similar meanings land near
each other. Retrieval is then: embed the question, find the nearest
paragraphs.

"Nearest" means **cosine similarity** — the angle between two vectors,
ignoring length, so a long paragraph and a short question stay comparable.

The vectors ship with the corpus, so this runs with no key and no network and
everyone compares the same numbers. **Embedding your own corpus is the first
Bonus Task**, and that is where cost and rate limits become real.

For Nadia, "embed the corpus" is the first entry in the page's cost
column: two thousand tickets go *through* an embedding model, once now and
again every time the chunking changes — and if that model is a vendor's
API, the tickets' names and account numbers go with them, which adds a
line to the cost column that is measured in months of vendor review, not
tokens.

> **🏢 In the enterprise:** an embedding model is a dependency with a
> version, and every vector in your index was produced by one specific
> version of it. Change the model — or let a vendor change it under you —
> and the query vectors no longer live in the same space as the stored
> ones, silently. The page's *"why was this record chosen?"* has no answer
> at all if the model that chose it is not recorded. The choice
> it forces is whether the model name and version are written into the
> index's metadata and checked at query time, or whether "search got worse
> last Tuesday" is a mystery someone spends a week on.

In [ ]:
vector_archive = np.load(DATA / "embeddings.npz", allow_pickle=False)
DOC_VECS, QUERY_VECS, RERANK_SCORES = vector_archive["doc"], vector_archive["query"], vector_archive["rerank"]

assert list(vector_archive["titles"]) == TITLES, (
    "embeddings.npz is out of step with corpus.jsonl"
)

print(
    f"✅ {DOC_VECS.shape[0]} document vectors, {DOC_VECS.shape[1]} dimensions each"
)
print(f"✅ already unit length: ‖v‖ = {np.linalg.norm(DOC_VECS[0]):.4f}")

In [ ]:
def cosine(query_vec, doc_matrix):
    """Cosine similarity of one vector against a matrix of vectors.

    Dot product over the product of the norms. These vectors are already
    normalised so the denominator is 1 -- which is why embedding APIs return
    normalised vectors, and why production dense retrieval is one matmul.
    """
    return (doc_matrix @ query_vec) / (
        np.linalg.norm(doc_matrix, axis=1) * np.linalg.norm(query_vec)
    )

def dense_rank(question_index: int) -> list[int]:
    """Document indices, best first."""
    return list(np.argsort(-cosine(QUERY_VECS[question_index], DOC_VECS)))

print(
    "a vector against itself:",
    round(float(cosine(DOC_VECS[0], DOC_VECS[:1])[0]), 4),
)

In [ ]:
ui.table(
    [
        {
            "rank": rank + 1,
            "paragraph": TITLES[doc_index],
            "gold": "⭐"
            if TITLES[doc_index] in QUESTIONS[0]["gold_titles"]
            else "",
        }
        for rank, doc_index in enumerate(dense_rank(0)[:8])
    ],
    title=f"dense — {QUESTIONS[0]['question']}",
)

One matrix multiply, and the stars tell you whether it worked — which is the
entire argument for having labels. Nadia's senior engineer found one wrong
diagnosis by reading. Labels find all of them by counting.

---
## Task 4 — BM25, from scratch

Before embeddings, search matched words. BM25 won that era and remains the
strongest simple baseline in information retrieval.

Three ideas, each present because the naive version fails without it:

| Idea | Without it |
| --- | --- |
| **IDF** — rare terms count for more | "the" contributes as much as "Maineiacs" |
| **Saturation** — the tenth mention adds less than the second | A page that repeats a word wins everything |
| **Length normalisation** — long documents do not win by being long | Every result is your longest document |

In [ ]:
import math
import re

def tokenise(text: str) -> list[str]:
    """Lowercase word tokens. Digits kept, so '2014' survives as a term."""
    return re.findall(r"[a-z0-9']+", text.lower())

CORPUS_TOKENS = [tokenise(text) for text in TEXTS]
corpus_size = len(CORPUS_TOKENS)
avg_doc_length = sum(len(tokens) for tokens in CORPUS_TOKENS) / corpus_size

doc_freq: dict[str, int] = {}
for doc_tokens in CORPUS_TOKENS:
    for term in set(doc_tokens):
        doc_freq[term] = doc_freq.get(term, 0) + 1

term_freqs = []
for doc_tokens in CORPUS_TOKENS:
    doc_term_freq: dict[str, int] = {}
    for term in doc_tokens:
        doc_term_freq[term] = doc_term_freq.get(term, 0) + 1
    term_freqs.append(doc_term_freq)

def bm25_rank(query: str, k1: float = 1.5, b: float = 0.75) -> list[int]:
    """Document indices, best first.

    k1 controls saturation, b controls length normalisation. These defaults
    are the ones everyone uses and almost nobody tunes.
    """
    scores = []
    for tf, toks in zip(term_freqs, CORPUS_TOKENS):
        total = 0.0
        for word in tokenise(query):
            if word not in tf:
                continue
            idf = math.log(
                1
                + (corpus_size - doc_freq[word] + 0.5)
                / (doc_freq[word] + 0.5)
            )
            total += (
                idf
                * (tf[word] * (k1 + 1))
                / (tf[word] + k1 * (1 - b + b * len(toks) / avg_doc_length))
            )
        scores.append(total)
    return list(np.argsort(-np.asarray(scores)))

print(
    f"✅ index over {corpus_size} documents, {len(doc_freq):,} distinct terms"
)

In [ ]:
ui.table(
    [
        {
            "rank": rank + 1,
            "paragraph": TITLES[doc_index],
            "gold": "⭐"
            if TITLES[doc_index] in QUESTIONS[0]["gold_titles"]
            else "",
        }
        for rank, doc_index in enumerate(bm25_rank(QUESTIONS[0]["question"])[:8])
    ],
    title=f"BM25 — {QUESTIONS[0]['question']}",
)

Forty lines, no model, no index to keep warm, and it is competitive. Hold that
thought — it returns in Task 6 in a way most people do not expect.

And look at what Nadia's engineers actually type: `INC-30417`, `OLT-17`,
`C-204`, `LOS`, *"CRC errors"*. Identifiers and exact terms. BM25 was built
for exactly that, and an embedding model blurs a port number into a
direction in space where every port number looks alike.

> **🏢 In the enterprise:** BM25 needs no model, no vendor, no data leaving
> the network, and no approval — it is forty lines of code you own. That
> is not a consolation prize. On Nadia's page it is the row with the
> shortest cost column — nothing to license, nothing to review, nothing
> leaving the building — and in a firm with a vendor-review queue
> the technique that ships is often the one that clears review first, and
> "I can show you the exact term that matched" clears review in a way "the
> model ranked it highly" does not. The choice it forces: before proposing
> an embedding model, measure whether the lexical baseline is already good
> enough.

---
## Task 5 — Hybrid

The standard advice: dense catches meaning, BM25 catches exact terms, so run
both and fuse. Best of each.

**Reciprocal Rank Fusion** is how, and it is one line: score each document by
the sum of `1 / (k + rank)` across the rankings it appears in. It uses ranks
rather than scores, so the two systems need no comparable scale — which is why
it beats more sophisticated fusion in practice.

Now measure it, rather than believing the paragraph above.

In [ ]:
def rrf(rankings: list[list[int]], n_docs: int, k: int = 60) -> list[int]:
    """Reciprocal Rank Fusion. `k` damps the influence of the very top ranks."""
    scores = np.zeros(n_docs)
    for ranking in rankings:
        for rank, doc_index in enumerate(ranking):
            scores[doc_index] += 1.0 / (k + rank + 1)
    return list(np.argsort(-scores))

RANKINGS = {}
for question_index, question_row in enumerate(QUESTIONS):
    dense_ranking = dense_rank(question_index)
    bm25_ranking = bm25_rank(question_row["question"])
    RANKINGS[question_index] = {
        "dense": dense_ranking,
        "bm25": bm25_ranking,
        "hybrid": rrf([dense_ranking, bm25_ranking], len(TITLES)),
    }

print(f"✅ three rankings for each of {len(RANKINGS)} questions")

In [ ]:
def complete_at_k(method: str, k: int) -> float:
    """Fraction of questions where EVERY gold paragraph is inside the top k.

    Not "did something relevant come back". Did enough come back to answer
    the question -- the only threshold a user cares about.
    """
    hits = 0
    for question_index, question in enumerate(QUESTIONS):
        top = {TITLES[doc_index] for doc_index in RANKINGS[question_index][method][:k]}
        hits += set(question["gold_titles"]) <= top
    return hits / len(QUESTIONS)

def recall_at_k(method: str, k: int) -> float:
    """Fraction of gold paragraphs found, averaged over questions."""
    return float(
        np.mean(
            [
                len(
                    set(question["gold_titles"])
                    & {TITLES[doc_index] for doc_index in RANKINGS[question_index][method][:k]}
                )
                / len(question["gold_titles"])
                for question_index, question in enumerate(QUESTIONS)
            ]
        )
    )

In [ ]:
ui.table(
    [
        {
            "technique": method,
            "recall@5": f"{recall_at_k(method, 5):.0%}",
            "complete answers @5": f"{complete_at_k(method, 5):.0%}",
        }
        for method in ("bm25", "dense", "hybrid")
    ],
    title="Measured on this corpus, top 5",
)

In [ ]:
ui.table(
    [
        {
            "question": question["question"][:40] + "…",
            **{
                method: str(
                    sorted(
                        RANKINGS[question_index][method].index(TITLES.index(title)) + 1
                        for title in question["gold_titles"]
                    )
                )
                for method in ("bm25", "dense", "hybrid")
            },
        }
        for question_index, question in enumerate(QUESTIONS)
    ],
    title="Rank of each gold paragraph",
)

**Hybrid did not beat its better parent.**

That is not what the standard advice predicts, and it is worth understanding
rather than explaining away. RRF fuses *ranks*, so a paragraph one system puts
1st and the other puts 46th lands in the middle. When both systems are good,
that averaging helps. When one is clearly stronger on your corpus, fusion drags
its correct answers down to meet the other's wrong ones.

The first question shows it exactly: dense puts the gold pair at 2 and 4, BM25
at 1 and 46, and fused they come out at 2 and 12 — outside the top five, so
the question goes from answerable to unanswerable.

> **Hybrid is a bet that your two retrievers fail on different questions.**
> When they fail on the same ones, or one is simply stronger, you have paid
> for two indexes and made the result worse. That is measurable in an
> afternoon and is almost never measured.

For Nadia the bet has a specific shape: BM25 should win the `OLT-17`
questions and lose the *"customer says the internet is slow at night"*
ones, and dense the reverse. If her labels show that split, hybrid earns
its second index and its second cost line. If BM25 wins both, the page
gets shorter, which her manager will notice before he notices anything
else.

---
### ❓ Question #2

You are about to measure BM25 and dense against ten questions you labelled
yourself. In one to three sentences, say what you would look for in the
*rank of each gold paragraph* table to know whether the two fail on
**different** questions — the only case in which fusing them helps — and
what you would conclude if they fail on the same ones.

**Answer:**

---
## Task 6 — The variable that matters more than the technique

Everything so far compared *techniques* with the embedding model held fixed.
Now hold the technique fixed and change the model.

These numbers come from `scripts/measure_retrieval.py --compare-models`, which
runs the same code you just wrote against three embedding models. It downloads
several gigabytes, so it is not in this notebook — but you can read what it
found.

In [ ]:
ui.table(
    [
        {
            "embedding model": "all-MiniLM-L6-v2",
            "params": "22M",
            "BM25": "4/10",
            "dense": "3/10",
            "hybrid": "3/10",
        },
        {
            "embedding model": "bge-small-en-v1.5",
            "params": "33M",
            "BM25": "4/10",
            "dense": "8/10",
            "hybrid": "5/10",
        },
        {
            "embedding model": "Qwen3-Embedding-4B",
            "params": "4,000M",
            "BM25": "4/10",
            "dense": "7/10",
            "hybrid": "5/10",
        },
    ],
    title="Complete answers in the top 5, by embedding model",
)

Three things there, and all three should change how you argue about retrieval.

**Dense is not reliably better than BM25.** With the first model it is *worse*
— 3/10 against 4/10. *"Use embeddings, they understand meaning"* is a claim
about a specific model on a specific corpus, and here it is false for one of
the three.

**A 33M model beat a 4,000M one.** A hundred and twenty times fewer
parameters. Whatever you are buying with the largest available embedding
model, on this corpus it is not accuracy.

**BM25 never moved.** No model to choose, no version to be surprised by, no
vendor. There is something to be said for the technique whose behaviour you
can predict — and, for Nadia, for the row in this table that does not
change when the approved embedding vendor changes next year.

> **The selection criterion, and the real lesson of this session:** the
> biggest lever is not which technique you pick. It is that you measured at
> all. Every table above took minutes to produce and contradicts advice that
> is repeated everywhere.

---
## Task 7 — Reranking, which is the fix

Here is the shape of the problem. Once you see it, the answer is obvious.

In [ ]:
ui.table(
    [
        {
            "k": top_k,
            "bm25": f"{complete_at_k('bm25', top_k):.0%}",
            "dense": f"{complete_at_k('dense', top_k):.0%}",
            "hybrid": f"{complete_at_k('hybrid', top_k):.0%}",
            "of retrieved text that is answer": f"{min(2 / top_k, 1.0):.0%}",
        }
        for top_k in (1, 3, 5, 10, 20, 50)
    ],
    title="Complete answers as you widen the net",
)

In [ ]:
# The same numbers as the table above. The table is the record; this is the
# shape -- and the shape is the argument, because "where does it stop
# improving" is a question about a curve and a reader should not have to
# reconstruct one from six rows to answer it.
k_values = [1, 3, 5, 10, 20, 50]
ui.lines(
    {method: [complete_at_k(method, top_k) for top_k in k_values] for method in ("bm25", "dense", "hybrid")},
    x=k_values,
    x_label="k (paragraphs retrieved)",
    y_label="complete answers",
    title="Where each retriever stops improving",
)

**Zero at k=1**, exactly as the labels predicted before any code ran.

Recall climbs as you widen the net and arrives somewhere around twenty — at
which point ninety percent of what you retrieved is not the answer. Send that
to a model and you pay for all of it on every request, and the model has to
find the needle you failed to.

So: **retrieve wide, then narrow with something more expensive.**

A **cross-encoder** is the something. A bi-encoder — everything in Task 3 —
embeds query and document *separately*, which is what makes it fast enough to
index millions of documents and also what stops it seeing how the two relate.
A cross-encoder reads them **together** and scores the pair.

Far too slow over a corpus. Exactly right over twenty candidates.

| | Bi-encoder | Cross-encoder |
| --- | --- | --- |
| Sees | Query and document separately | Both at once |
| Precompute | Yes — embed the corpus once | No — every pair, every query |
| Cost | One matrix multiply | One model call per candidate |
| Its job | Do not miss anything | Put the right thing first |

The scores ship with the corpus, from `BAAI/bge-reranker-v2-m3`.

In [ ]:
def rerank(
    question_index: int,
    method: str = "hybrid",
    first_stage: int = 20,
    keep: int = 5,
) -> list[int]:
    """Retrieve `first_stage` cheaply, score each pair, keep `keep`."""
    candidates = RANKINGS[question_index][method][:first_stage]
    return sorted(
        candidates, key=lambda doc_index: -RERANK_SCORES[question_index][doc_index]
    )[:keep]

RERANK_COMPLETE = [
    float(set(question["gold_titles"]) <= {TITLES[doc_index] for doc_index in rerank(question_index)})
    for question_index, question in enumerate(QUESTIONS)
]
print(
    f"complete answers after reranking: {int(sum(RERANK_COMPLETE))}/{len(QUESTIONS)}"
    f"  ({np.mean(RERANK_COMPLETE):.0%})"
)

In [ ]:
ui.table(
    [
        {
            "question": question["question"][:44] + "…",
            "hybrid top-5": "✅"
            if set(question["gold_titles"])
            <= {TITLES[doc_index] for doc_index in RANKINGS[question_index]["hybrid"][:5]}
            else "✗",
            "reranked top-5": "✅"
            if set(question["gold_titles"]) <= {TITLES[doc_index] for doc_index in rerank(question_index)}
            else "✗",
        }
        for question_index, question in enumerate(QUESTIONS)
    ],
    title="Same twenty candidates, reordered",
)

Nothing new was retrieved. The reranker saw exactly the twenty paragraphs
hybrid already had — it just put the right ones in the top five.

**The information was already there and the ranking was throwing it away.**
That is the most common recoverable failure in a production retrieval system,
and it is invisible unless you measure completeness rather than relevance.

> **If you take one thing from this session to work on Monday, take this one.**
> Retrieve 20–50 with something cheap, rerank to 5 with a cross-encoder. A few
> lines, no new infrastructure, and here it is the difference between half your
> questions and all of them.

**The cost, stated honestly:** one model call per candidate, so twenty
candidates is twenty forward passes of a small model — tens of milliseconds on
a GPU, noticeably more on a CPU, and it is per query rather than per corpus.
Measure it against your latency budget before promising it.

> **🏢 In the enterprise:** a reranker is the first component in this
> pipeline whose cost is *per query* rather than *per corpus*, so it is the
> first one that needs a latency budget and a place to run. A cross-encoder
> is a model, it has to execute somewhere, and if that somewhere is a
> hosted API then twenty of Nadia's tickets — names, addresses, account
> numbers — leave the network per query, which puts a vendor review on
> the page next to the latency. The choice it forces
> is where it lives — in the request path on a CPU, where twenty
> candidates might add half a second; behind its own endpoint on a GPU
> inside the network, which is a service somebody has to own; or not at
> all, with a smaller first stage. Log the reranker's time per request
> separately from the retrieval time, because when the P99 moves, that is
> the number that says which half moved.

---
### ❓ Question #3

Your endpoint will have a latency budget, whether or not anyone has written
it down yet. In one to three sentences, say where twenty cross-encoder
calls per query fit inside it — or do not — and what first-stage `k` you
would start with for your corpus, given that the reranker can only promote
what the first stage retrieved.

**Answer:**

---
## Task 8 — Agentic retrieval, when one shot is not enough

Everything so far has been **one shot**: take the question, retrieve, stop.
That is a strange constraint if you think about how a person searches.

Someone answering *"the director of Big Stone Gap is based in what New York
city?"* looks up the film, learns the director is Adriana Trigiani, and
**searches again**. Two searches — and the second is only possible after
reading the first result. Nadia, asked why the fibre drops, opens the
ticket, reads `OLT-17` in the symptom, and searches for *that*. Same
shape.

That is agentic retrieval: give the model search tools and let it drive. It
subsumes the multi-hop problem, because the second hop is just another search.

In [ ]:
def search(query: str, k: int = 3) -> list[dict]:
    """Keyword search over the corpus. A tool a model can call."""
    wanted = set(tokenise(query))
    scored = []
    for doc_index, text in enumerate(TEXTS):
        overlap = len(wanted & set(tokenise(text)))
        if overlap:
            scored.append((overlap, doc_index))
    # Sort on overlap descending, then on index ASCENDING -- the second key
    # matters more than it looks. `scored.sort(reverse=True)` reverses both,
    # so ties went to whichever document happened to sit later in a corpus
    # that is stored alphabetically. Ties are common when the score is a
    # count of shared words, and the earlier version of this cell let an
    # alphabetical accident pick the answer for two of its own examples.
    scored.sort(key=lambda pair: (-pair[0], pair[1]))
    return [
        {"title": DOCS[doc_index]["title"], "opening": DOCS[doc_index]["text"][:140]}
        for _, doc_index in scored[:k]
    ]

def read_page(title: str) -> str:
    """Fetch one paragraph in full. The other half of what an agent needs."""
    for doc in DOCS:
        if doc["title"].lower() == title.lower():
            return doc["text"]
    return f"no page titled {title!r}"

print("✅ two tools: search, read_page")
print(
    "   search('Big Stone Gap director') ->",
    search("Big Stone Gap director")[0]["title"],
)

Two tools, twenty lines. Walk the loop by hand before giving it to a model —
the point is the *sequence*, not the model's cleverness.

In [ ]:
first_hit = search("Big Stone Gap film director")[0]
first_page = read_page(first_hit["title"])
second_hit = search("Adriana Trigiani")[0]
second_page = read_page(second_hit["title"])

ui.table(
    [
        {
            "step": "1 · search",
            "for": "Big Stone Gap film director",
            "result": first_hit["title"],
        },
        {
            "step": "2 · read",
            "for": first_hit["title"],
            "result": first_page[:66] + "…",
        },
        {
            "step": "3 · search",
            "for": "Adriana Trigiani  ← learned in step 2",
            "result": second_hit["title"],
        },
        {
            "step": "4 · read",
            "for": second_hit["title"],
            "result": second_page[:66] + "…",
        },
    ],
    title="The second search is only possible after reading the first result",
)

Step 3 is the whole idea. *"Adriana Trigiani"* is not in the question — it was
learned in step 2 — so **no one-shot retriever can search for it**, whatever
its ranking function.

### An aside that is not an aside: who wins a tie

`search` scores by counting shared words, so ties are not an edge case — they
are most of the ranking. Two documents share two query words, and something
has to break the tie. That something is `sort`.

This cell used to sort with `reverse=True`, which reverses *both* keys: the
score and the index it was paired with. Ties therefore went to whichever
paragraph sat later in the corpus, and the corpus is stored alphabetically. So
a query for `"Adriana Trigiani"` tied at two words between her own page and
the film's — and returned the film, because *B* sorts after *A*.

**The demo above claimed a hop had happened when it had not**, and nothing
errored, because a plausible document came back either way.

> A tie-break is a ranking decision you did not know you were making. When a
> retriever returns something defensible-but-wrong, look at how many results
> share the top score before you look at anything else — and if the answer is
> "several", your scoring function is not the thing choosing.

This is why the measured comparisons earlier used ranking functions that
produce real-valued scores. Ties are rare in those. They are everywhere in a
word count.

### What it buys, and what it costs

| | One shot | Agentic |
| --- | --- | --- |
| Model calls | none for retrieval | one per hop, plus deciding to stop |
| Latency | a matrix multiply | seconds, and variable |
| Multi-hop | structurally impossible | natural |
| Debugging | one ranking to inspect | a trajectory, which is harder |
| Failure mode | the wrong passage | loops, or stops early and is confident |

**The honest comparison:** reranking got every question here for twenty
forward passes. Agentic retrieval also solves them, for several model calls and
unpredictable latency. On a corpus this shape the reranker is the better
engineering. On questions with three or four hops, or where the corpus must be
filtered before it is searched, the loop is the only thing that works.

Reach for it when one shot has been *measured* to be insufficient — not because
it is the more interesting architecture.

> **Giving a model `search` and `read_page` over your own documents is also the
> cheapest useful thing on this list.** No index, no embedding model, no
> re-indexing when a document changes, and every step is legible to a security
> reviewer. For a few hundred documents it is frequently the right answer, and
> it is the technique enterprise teams most often never try.

For Nadia it fills two lines on the page at once. Every `read_page` call
is a ticket id in the trace, so "every claim cites its source" is free.
And `search` is *her* function, so the rule that an engineer sees only
their own region's customers is one line inside it — the model cannot ask
for a ticket the function will not return. The loop is the one technique
here where the access rule lives in code she wrote rather than in a filter
she has to remember to apply.

---
## Task 9 — Choose, and say why

Everything on one screen.

In [ ]:
ui.table(
    [
        {
            "technique": "BM25",
            "complete @5": f"{complete_at_k('bm25', 5):.0%}",
            "setup": "an index",
            "per query": "microseconds",
            "when it wins": "exact terms and identifiers; no model to get approved",
        },
        {
            "technique": "dense",
            "complete @5": f"{complete_at_k('dense', 5):.0%}",
            "setup": "embed the corpus",
            "per query": "one matrix multiply",
            "when it wins": "paraphrase — if the model suits your text",
        },
        {
            "technique": "hybrid",
            "complete @5": f"{complete_at_k('hybrid', 5):.0%}",
            "setup": "both indexes",
            "per query": "both",
            "when it wins": "when the two fail on different questions. Measure it",
        },
        {
            "technique": "rerank",
            "complete @5": f"{np.mean(RERANK_COMPLETE):.0%}",
            "setup": "a cross-encoder",
            "per query": "k model calls",
            "when it wins": "almost always. The cheapest large win here",
        },
        {
            "technique": "agentic",
            "complete @5": "different shape",
            "setup": "two functions",
            "per query": "several model calls",
            "when it wins": "many hops, or a corpus too small to justify an index",
        },
    ],
    title="The comparison — your numbers, from this corpus",
)

### The questions that actually decide it

1. **How many documents, really?** Under a few hundred, an agent with `search`
   and `read_page` beats an index on accuracy, cost, auditability, and how
   fast you can get it approved. Most corpora are far smaller than people
   assume.
2. **How many of your questions need more than one document?** If the answer
   is "most", top-k is the wrong shape and tuning will not fix it.
3. **What shape are the questions?** Look at ten real ones. Identifiers mean
   lexical matching. Paraphrase means embeddings — and means measuring *which*
   embedding model.
4. **How often does the corpus change?** Re-indexing is the cost people forget,
   and the reason systems go quietly stale.
5. **Who has to trust the answer?** "I can show you the exact line" beats "the
   model ranked it highly", sometimes decisively.
6. **What can you actually get approved?** The best technique you cannot deploy
   is worth less than the adequate one you can. The *setup* column above
   is the answer: an index is code, an embedding model is a vendor, a
   cross-encoder is a service, and each of those is a different length of
   calendar.

Nadia's page, for the record: two thousand tickets and eighty runbooks;
most questions need two; identifiers everywhere; the corpus grows by forty
tickets a day; the diagnosis goes on the ticket so it has to cite; and
the thing that fits the budget and the calendar this quarter is forty
lines of BM25 and two functions, running on the service desk's own
server. *What would change her mind:* paraphrase questions passing a fifth
of traffic, or the labels showing BM25 and dense failing on different
tickets. That is not the most interesting architecture in the table. It
is the one her manager signs.

### When a numpy array stops being enough

Everything above is a matrix and a sort. For a hundred paragraphs — and for a
surprising number of real corpora — that is the right answer. An index adds a
dependency, a service, an approval conversation, and a re-indexing story.

You have outgrown it when the vectors no longer fit comfortably in memory, when
you need metadata filtering applied **before** ranking, or when more than one
process needs the same index. Hundreds of thousands of documents, not
thousands.

Two worth knowing, suiting different situations:

| | Shape | Why you would pick it |
| --- | --- | --- |
| **Chroma** | `pip install chromadb`, persists to a local directory | The right first move. No service to run, drops straight into a prototype, small enough to learn in a sitting |
| **Qdrant** | Runs as a container | If you are comfortable with Docker it deploys the way everything else you own deploys. Stronger filtering and a real scaling story |

Neither is a default, and reaching for one before measuring an array is how
teams end up operating a database to serve four hundred documents.

```python
# Chroma — about as much code as the numpy version
import chromadb

client = chromadb.PersistentClient(path="./index")
collection = client.get_or_create_collection("corpus")
collection.add(ids=titles, documents=texts, embeddings=doc_vecs.tolist())

hits = collection.query(query_embeddings=[query_vec.tolist()], n_results=5)
```

What that gives you which the array does not is the `where=` argument —
filtering on metadata *before* the search rather than after, which you will
need the first time your documents have access levels. Nadia's already
do: an engineer sees their own region's customers and nothing else, so the
rule is `where={"region": "north"}`, applied before a single similarity is
computed. Filtering *after* ranking means another region's customer was
scored, ranked, and possibly logged before they were dropped — and that is
a line on the page her manager cannot sign, whatever the other three
numbers say.

---
### ❓ Question #4

Six questions above decide the technique. For *your* corpus and *your*
users, name the one that decides it — and in one to three sentences, say
what you can actually get approved at your firm, because the best
technique you cannot deploy is worth less than the adequate one you can.

**Answer:**

---
# Conclusion

You measured five ways of finding the right paragraph against the same
labelled questions, and the numbers disagreed with the usual advice more
than once: hybrid lost to its better parent, a 33M embedding model beat a
4B one, and the largest single improvement came from reordering what had
already been retrieved. The technique matters less than the fact that you
measured.

Nadia's page, filled in:

| The page asks | What goes there |
| --- | --- |
| **How often is it right?** | Complete diagnoses at *k*, per technique, on labelled tickets — the number that was zero at top-1 and moved most when the reranker reordered what was already there |
| **What does it cost?** | Per corpus for embeddings, per query for the reranker, and a calendar for anything that is a vendor or a service; BM25's line is the shortest |
| **How long does a ticket wait?** | Microseconds for an index, seconds and variable for a loop, and the reranker's time logged on its own so the SLA can be defended |
| **What would change your mind?** | The split in the rank-of-each-gold table; the share of paraphrase questions; the corpus outgrowing an array |

| Component | What We Built | What Production Looks Like |
| --- | --- | --- |
| Index | A matrix and a sort | An ANN index — approximate, and the approximation is a tuning knob |
| Labels | Ten labelled questions | Hundreds, refreshed as the corpus drifts, tracked per release |
| Embeddings | Vectors that shipped with the corpus | Vectors produced by a pinned model version, recorded in the index metadata and checked at query time |
| Reranker | Precomputed cross-encoder scores | A reranker behind its own endpoint, with its own latency budget and its own timing in the trace |
| Chunking | The chunks the dataset chose | A chunk size you measured, and re-measure when the documents change |
| Access | One corpus, one access level | Metadata filtering applied before ranking, because documents have owners |
| Tools | `search` over everything | The same tool, scoped to what this user is allowed to see |

The last row is the one that gets missed until a security review finds it,
and the one line on Nadia's page that no measurement can earn back.

---
# Additional Context

- [Reciprocal Rank Fusion](https://cormack.uwaterloo.ca/cormacksigir09-rrf.pdf) — Cormack et al., 2009. Six pages, one formula, and it outperforms considerably more complicated fusion methods. The best effort-to-value ratio on this list
- [The Probabilistic Relevance Framework: BM25 and Beyond](https://www.staff.city.ac.uk/~sbrp622/papers/foundations_bm25_review.pdf) — Robertson & Zaragoza. Where the forty lines in Task 4 come from; read §3 for the intuition
- [Dense Passage Retrieval](https://arxiv.org/abs/2004.04906) — Karpukhin et al., 2020. The paper that made dense retrieval the default; worth reading for what it measures
- [sentence-transformers — Cross-Encoders](https://sbert.net/docs/cross_encoder/usage/usage.html) — `CrossEncoder.rank(query, passages)` is the live version of Task 7's precomputed scores, in three lines
- [LiteLLM — function calling](https://docs.litellm.ai/docs/completion/function_call) — how to hand Task 8's `search` and `read_page` to a model so it drives the loop itself, which this notebook walked by hand
- [Anthropic — Contextual Retrieval](https://www.anthropic.com/news/contextual-retrieval) — a practical fix for the context a chunk loses when cut from its document
- [Chroma — Evaluating Chunking Strategies](https://www.trychroma.com/research/evaluating-chunking) — the measurement most teams skip, useful for how it constructs the evaluation
- [Agentic Retrieval-Augmented Generation: A Survey on Agentic RAG](https://arxiv.org/abs/2501.09136) — Singh et al., 2025. A *pre-read for this session*: the taxonomy of loops a model can drive, of which Task 8's two-tool walk is the smallest
- [GraphRAG](https://www.microsoft.com/en-us/research/project/graphrag/) — Microsoft Research. The other *pre-read*: when the question is about the whole corpus rather than a passage in it — *"which cabinets generate the most repeat tickets"* — and neither top-k nor a loop is the right shape
- **Keep building:** close the loop in Task 8 — give a model the two tools, cap it at three searches, make it say when it has not found the answer, and score it against the same ten labels as everything else. Then you have six rows in the comparison table instead of five, and the sixth is the one nobody measures. Put Nadia's region scope inside `search` while you are there, and confirm that a query for another region's customer returns nothing rather than a near miss.

---
# Bonus Tasks

Optional. The first is worth doing if you have the hour, and it is the one
that turns this notebook into your own numbers; the rest are meaningfully
harder.

### Your corpus, your questions, your numbers

This corpus has labels because somebody paid people to write them. Yours does
not, and neither does Nadia's — that is the real difference between this
notebook and Monday.

**Make your own labels. Twenty minutes, and it is the highest-value twenty
minutes here.**

1. Take ten real questions from your use case — from a ticket, a chat log, a
   colleague. Not invented ones.
2. For each, write down which document *should* answer it. If two are needed,
   write both. That is your `gold_titles`.
3. Run `complete_at_k` against your own corpus.

You now have recall@k on your own data, which is more than most teams running
retrieval in production have ever had.

> **That file is also an eval set.** Ten questions with labelled answers is the
> retrieval half of a harness, and it took twenty minutes. Keep it.

### Embedding your own corpus

The vectors here were precomputed. Yours will not be, and this is where
retrieval stops being free:

In [ ]:
run_embed = mo.ui.run_button(
    label="Embed five sample strings through my endpoint"
)
run_embed

In [ ]:
mo.stop(
    not run_embed.value,
    mo.md(
        "Press the button to embed five short strings through your configured "
        "model — enough to see the shape of the call and what it costs."
    ),
)

from litellm import embedding

sample_texts = [
    "the quarterly revenue report for the northern region",
    "how do I reset a locked account",
    "escalation policy for a sev-1 incident",
    "who owns the billing service",
    "what changed in the retention policy last year",
]

# An explicit timeout, always. An inherited library default is how a call
# hangs for forty minutes and returns nothing.
#
# No streaming here, and not by oversight: an embedding endpoint returns
# one vector per input when it is finished. There are no intermediate
# tokens to show. Batch size, not streaming, is the lever on how long
# you wait.
#
# `embed_kwargs()` rather than model= and api_base= by hand, because the
# embedding endpoint often sits on a different host from the chat one and
# therefore often needs a different credential.
try:
    embed_response = embedding(**CFG.embed_kwargs(), input=sample_texts, timeout=600)
except Exception as error:
    # One failure here is worth naming rather than raising, because it is
    # both common and misleading: a server can list an embedding model in
    # /v1/models and still refuse to embed. llama.cpp does exactly that
    # unless it was started with `--embeddings`.
    embed_response = None
    error_detail = f"{type(error).__name__}: {error}"
    if "501" in error_detail or "does not support embeddings" in error_detail:
        print("❌ The endpoint answered, and said it does not do embeddings.")
        print("   A model appearing in /v1/models is not a promise that the")
        print("   route exists — serving chat and serving embeddings are")
        print("   separate switches on the same server.")
    else:
        print(f"❌ Embedding call failed: {error_detail[:200]}")
    print()
    print("   Write it in use_case/ecosystem.md and carry on. The vectors for")
    print("   this corpus are committed, so nothing above depended on it.")

if embed_response is not None:
    sample_vectors = np.asarray(
        [item["embedding"] for item in embed_response.data], dtype=np.float32
    )

    print(f"✅ {sample_vectors.shape[0]} vectors, {sample_vectors.shape[1]} dimensions")
    print(
        f"   already normalised: "
        f"{bool(np.allclose(np.linalg.norm(sample_vectors, axis=1), 1.0, atol=1e-3))}"
    )
    print(f"   usage: {getattr(embed_response, 'usage', None)}")
    print()
    print(
        "Now scale it. A 50,000-document corpus at ~200 tokens each is 10M "
        "tokens — once for the corpus, and again every time you change the "
        "chunk size. That number, not the accuracy, is usually what decides "
        "the architecture."
    )

> **If that cell failed**, your embedding model is not configured or not
> reachable — which is itself worth writing down. Everything above still runs:
> the vectors are committed precisely so a blocked endpoint costs you one cell
> rather than the session.

---
### Harder

- **Tune the first stage against the reranker.** Retrieve 50 instead of 20 —
  does the extra recall survive to the top five, and what does it cost?
- **Break the reranker.** Find a question where it puts a distractor first.
  They exist, and knowing what they look like is worth more than the score.
- **Chunk to sentences and re-measure.** The retrieval metric will improve.
  Check whether the answers do.
- **Give the agent a budget.** Cap it at three searches and make it say when it
  has not found the answer. Confidently wrong is worse than "I don't know".
- **Filter before you rank.** Tag each paragraph with a pretend access level and
  make retrieval respect it. Notice that filtering *after* ranking gives you a
  shorter list and a security incident — the line on Nadia's page that
  no other number can offset.
- **Measure on a replica when you cannot measure on the real thing.** Nadia
  could not run this notebook on real tickets until access was granted.
  Two ways
  through, both worth trying on your own export: *mask* the identifiers
  in a real export (ticket ids, hostnames, names → stable placeholders,
  one mapping file kept inside the network) and re-run the three
  techniques on the masked copy; or *generate* a replica corpus from your
  schema and a written description, with no real records as seeds, and
  label ten questions on that. Then compare: which findings survive the
  swap back to real data, and which were artefacts of the stand-in?
  The ones that survive are the ones you can put in `decisions.md`.